# ROQ Parameter Estimation for GW170817

Parameter estimation using the **mlgw\_bns\_jax** surrogate waveform model
with **Reduced Order Quadrature (ROQ)** likelihood acceleration and
the **SHARPy** SMC sampler.

The ROQ basis reduces the frequency-domain inner products from ~253k to
O(1000) evaluation points, giving ~100x speedup in likelihood evaluation.

**Prerequisites:** Run `build_roq_basis.ipynb` (or `build_roq_basis.py`)
first to generate the ROQ interpolants.

In [ ]:
import os, subprocess, sys

COLAB = "google.colab" in sys.modules
REPO_DIR = "/content/mlgw_bns_jax" if COLAB else os.getcwd()

if COLAB:
    # ── Install JAX with CUDA 12 support ─────────────────────────────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "jax[cuda12]",
        "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "corner", "gwpy", "h5py", "ripplegw", "lalsuite", "jaxopt", "netket",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--force-reinstall", "--no-deps",
        "blackjax @ git+https://github.com/gabrieledemasi/blackjax@main",
    ])

    if not os.path.isdir(REPO_DIR):
        subprocess.check_call([
            "git", "clone", "--branch", "blackjax_ns_gw_pe", "--depth", "1",
            "https://github.com/jacopok/mlgw_bns.git", REPO_DIR,
        ])

    sharpy_repo = os.path.join(REPO_DIR, "_sharpy_repo")
    sharpy_pkg  = os.path.join(sharpy_repo, "sharpy")
    sharpy_link = os.path.join(REPO_DIR, "sharpy")
    if not os.path.isdir(sharpy_repo):
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/gabrieledemasi/sharpy.git",
            sharpy_repo,
        ])
    if not os.path.exists(sharpy_link):
        os.symlink(sharpy_pkg, sharpy_link)

    # ── Fix blackjax circular import ─────────────────────────────────
    import site
    for _sp in site.getsitepackages():
        _chees = os.path.join(_sp, "blackjax", "adaptation", "chees_adaptation.py")
        if os.path.exists(_chees):
            with open(_chees, "r") as f:
                _csrc = f.read()
            _old_ci = "import blackjax.optimizers.dual_averaging as dual_averaging"
            _new_ci = "from blackjax.optimizers import dual_averaging"
            if _old_ci in _csrc:
                _csrc = _csrc.replace(_old_ci, _new_ci)
                with open(_chees, "w") as f:
                    f.write(_csrc)
                print("Patched chees_adaptation.py")
            break

    # ── Fix ripplegw import in SHARPy ────────────────────────────────
    _gw_lik = os.path.join(sharpy_pkg, "GW_likelihood.py")
    _old_import = "from ripplegw import ms_to_Mc_eta"
    with open(_gw_lik, "r") as f:
        _src = f.read()
    if _old_import in _src and "try:" not in _src.split(_old_import)[0][-30:]:
        _new_import = (
            "try:\n"
            "    from ripplegw import ms_to_Mc_eta\n"
            "except ImportError:\n"
            "    def ms_to_Mc_eta(m):\n"
            "        m1, m2 = m\n"
            "        return (m1 * m2) ** (3 / 5) / (m1 + m2) ** (1 / 5), m1 * m2 / (m1 + m2) ** 2"
        )
        _src = _src.replace(_old_import, _new_import)
        with open(_gw_lik, "w") as f:
            f.write(_src)
        print("Patched GW_likelihood.py")

    os.chdir(REPO_DIR)
    print(f"Working directory: {os.getcwd()}")
else:
    print("Not running on Colab -- skipping setup.")

In [ ]:
import os, sys, time, shutil
import numpy as np

_GPS_START = 1187008114
_DURATION  = 1024
_SRATE     = 4096
_DATA_DIR  = "gw170817_data"
os.makedirs(_DATA_DIR, exist_ok=True)

_DETECTORS = ["H1", "L1", "V1"]

_DCC_GWF_URL = (
    "https://dcc.ligo.org/public/0144/T1700406/003/"
    "L-L1_CLEANED_HOFT_C02_T1700406_v3-1187008667-4096.gwf"
)
_DCC_CHANNEL = "L1:DCH-CLEAN_STRAIN_C02_T1700406_v3"
_DCC_GPS0    = 1187008667
_DCC_SRATE   = 16384


def _ensure_gwf_backend():
    for mod in ("frameCPP", "lalframe", "framel"):
        try:
            __import__(mod)
            return
        except ImportError:
            pass
    import subprocess
    for pkg in ("framel",):
        ret = subprocess.call([sys.executable, "-m", "pip", "install", "-q", pkg])
        if ret == 0:
            return
    raise ImportError("Cannot read GWF files. Install framel.")


def _read_gwf_channel(path, channel, start, end):
    try:
        from gwpy.timeseries import TimeSeries
        ts = TimeSeries.read(path, channel, start=start, end=end)
        return np.asarray(ts.value, dtype=np.float64), float(ts.sample_rate.value)
    except Exception:
        pass
    import framel
    vec = framel.frgetvect1d(path, channel, start, end - start, 0)
    return np.asarray(vec[0], dtype=np.float64), 1.0 / vec[3]


_all_exist = all(
    os.path.isfile(os.path.join(_DATA_DIR,
        f"{d[0]}-{d}_BWCLEANED_4KHZ-{_GPS_START}-{_DURATION}.txt"))
    for d in _DETECTORS
)

if _all_exist:
    print("Cleaned data files already exist.")
else:
    from gwpy.timeseries import TimeSeries
    from scipy.signal import decimate as _decimate

    for det in _DETECTORS:
        out_file = os.path.join(_DATA_DIR,
            f"{det[0]}-{det}_BWCLEANED_4KHZ-{_GPS_START}-{_DURATION}.txt")

        if det == "L1":
            print("L1: building cleaned timeseries")
            ts_raw = TimeSeries.fetch_open_data("L1", _GPS_START, _GPS_START + _DURATION, sample_rate=_SRATE)
            gwf_local = os.path.join(_DATA_DIR, "L1_cleaned_bw_T1700406.gwf")
            if not os.path.isfile(gwf_local):
                import requests
                print("  Downloading BayesWave GWF from DCC...", flush=True)
                resp = requests.get(_DCC_GWF_URL, stream=True)
                resp.raise_for_status()
                with open(gwf_local, "wb") as fout:
                    for chunk in resp.iter_content(chunk_size=1 << 20):
                        fout.write(chunk)
            _ensure_gwf_backend()
            _need_end = _GPS_START + _DURATION
            bw_data, bw_sr = _read_gwf_channel(gwf_local, _DCC_CHANNEL, _DCC_GPS0, _need_end)
            if int(round(bw_sr)) != _SRATE:
                factor = int(round(bw_sr)) // _SRATE
                bw_data = _decimate(bw_data, factor, ftype="iir", zero_phase=True)
            n_raw = int((_DCC_GPS0 - _GPS_START) * _SRATE)
            strain = np.concatenate([ts_raw.value[:n_raw], bw_data])
            with open(out_file, "w") as fw:
                fw.write(f"# BayesWave-cleaned L1 strain GW170817\n# {_SRATE} Hz\n")
                for val in strain:
                    fw.write(f"{val:.16e}\n")
            print("  L1 done.")
        else:
            print(f"{det}: downloading from GWOSC...", flush=True)
            ts = TimeSeries.fetch_open_data(det, _GPS_START, _GPS_START + _DURATION, sample_rate=_SRATE)
            with open(out_file, "w") as fw:
                fw.write(f"# {det} raw GWOSC strain GW170817\n# {_SRATE} Hz\n")
                for val in ts.value:
                    fw.write(f"{val:.16e}\n")
            print(f"  {det} done.")
    print("All detectors ready.")

In [ ]:
from __future__ import annotations
import os, sys, time
from functools import partial
import numpy as np

if "google.colab" not in sys.modules:
    os.environ.setdefault("JAX_PLATFORMS", "cpu")

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
print("JAX devices:", jax.devices())

In [ ]:
from jax_import_n_predict import load_predict

MODEL_PATH = "mlgw_bns_jax_model.h5"
_mlgw_predict = load_predict(MODEL_PATH)

import sharpy.GW_likelihood as _gw_mod
from sharpy.utils import McQ2Masses


def _template_mlgw_bns(params, frequency_array):
    mc, q = params[6], params[7]
    m1, m2 = McQ2Masses(mc, q)
    total_mass = m1 + m2
    chi1, chi2 = params[9], params[10]
    lambda_1, lambda_2 = params[11], params[12]
    phic = params[4]
    dist_mpc = jnp.exp(params[2])
    inclination = params[3]

    mlgw_params = jnp.array([q, lambda_1, lambda_2, chi1, chi2])
    hp, hc = _mlgw_predict(
        mlgw_params, frequency_array,
        total_mass=total_mass,
        distance_mpc=dist_mpc,
        inclination=inclination,
    )
    phase_factor = jnp.exp(-1j * phic)
    return hp * phase_factor, hc * phase_factor


_gw_mod.template = _template_mlgw_bns

from sharpy.GW_likelihood import GWNetwork, log_likelihood_det
from sharpy.smc_functions import run_sharpy
import sharpy.PSDs

print("Model loaded. SHARPy template patched with mlgw_bns_jax.")

In [ ]:
TRIGGER_TIME     = 1187008882.43
SEGMENT_DURATION = 128.0
SAMPLING_RATE    = 4096
F_LOWER          = 23.0
F_UPPER          = 2000.0
DATA_START_GPS   = 1187008114
DATA_DURATION    = 1024

FIXED_RA  = 3.44616     # rad (NGC 4993)
FIXED_DEC = -0.408084   # rad

DATA_DIR = "gw170817_data"
OUTDIR   = "outdir_GW170817_roq"
LABEL    = "GW170817_roq_pe"
os.makedirs(OUTDIR, exist_ok=True)

print(f"Segment: {SEGMENT_DURATION}s  ->  df = {1/SEGMENT_DURATION:.4f} Hz")
print(f"Fixed sky: RA={FIXED_RA:.5f}, Dec={FIXED_DEC:.6f} (NGC 4993)")

In [ ]:
data_files = {
    "H1": os.path.join(DATA_DIR, f"H-H1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "L1": os.path.join(DATA_DIR, f"L-L1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "V1": os.path.join(DATA_DIR, f"V-V1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
}
for det, f in data_files.items():
    assert os.path.isfile(f), f"Missing: {f}"
    print(f"{det}: {os.path.basename(f)}")

detector_settings = {}
for det in ["H1", "L1", "V1"]:
    detector_settings[det] = dict(
        data_file=data_files[det], channel="GWOSC",
        trigger_time=TRIGGER_TIME, duration=SEGMENT_DURATION,
        sampling_rate=SAMPLING_RATE,
        f_lower=F_LOWER, f_upper=F_UPPER,
        psd_file=None, psd_method="welch",
        download_data=False, zero_noise=False,
    )

print(f"\nBuilding GW network (segment={SEGMENT_DURATION}s)...")
t0 = time.time()
gw_network = GWNetwork(detector_settings, injection_parameters=None)
print(f"Network built in {time.time() - t0:.2f} s")

## Load ROQ basis

Load the empirical interpolants and frequency nodes produced by `build_roq_basis.py`.
These allow us to evaluate inner products on a compressed frequency grid.

In [ ]:
ROQ_DIR = "./roq_basis_mlgw_bns_jax/ROQ_data"

# ── Linear basis: for <d|h> inner products ────────────────────────────
B_lin     = jnp.array(np.load(os.path.join(ROQ_DIR, "linear", "basis_interpolant_linear.npy")))
nodes_lin = np.load(os.path.join(ROQ_DIR, "linear", "empirical_nodes_linear.npy"))
f_lin     = np.load(os.path.join(ROQ_DIR, "linear", "empirical_frequencies_linear.npy"))

# ── Quadratic basis: for <h|h> inner products ────────────────────────
B_qua     = jnp.array(np.load(os.path.join(ROQ_DIR, "quadratic", "basis_interpolant_quadratic.npy")))
nodes_qua = np.load(os.path.join(ROQ_DIR, "quadratic", "empirical_nodes_quadratic.npy"))
f_qua     = np.load(os.path.join(ROQ_DIR, "quadratic", "empirical_frequencies_quadratic.npy"))

# Full frequency grid
deltaF = 1.0 / SEGMENT_DURATION
f_full = np.arange(F_LOWER, F_UPPER + deltaF, deltaF)

print(f"Full frequency grid : {len(f_full)} points")
print(f"Linear ROQ nodes    : {len(nodes_lin)} ({len(f_full)//len(nodes_lin)}x speedup)")
print(f"Quadratic ROQ nodes : {len(nodes_qua)} ({len(f_full)//len(nodes_qua)}x speedup)")
print(f"B_lin shape: {B_lin.shape}")
print(f"B_qua shape: {B_qua.shape}")

## ROQ Likelihood

The standard GW log-likelihood is:

$$\log \mathcal{L} = -\frac{1}{2} \sum_k \frac{|d_k - h_k|^2}{S_n(f_k)} \Delta f$$

Expanding: $|d - h|^2 = |d|^2 - 2\Re\langle d | h \rangle + \langle h | h \rangle$

With ROQ we replace:
- $\langle d | h \rangle \approx \sum_j w_j^{\text{lin}} \, d(f_j^{\text{lin}})^* \, h(f_j^{\text{lin}})$ using the **linear** basis
- $\langle h | h \rangle \approx \sum_j w_j^{\text{qua}} \, |h(f_j^{\text{qua}})|^2$ using the **quadratic** basis

The ROQ weights $w_j$ are precomputed from the basis interpolant and the PSD.

In [ ]:
# ── Precompute ROQ weights per detector ────────────────────────────────
batched_det = gw_network.batched_detector
n_det = len(batched_det.Frequency)

roq_weights_lin = []
roq_weights_qua = []
data_lin_proj   = []

df = deltaF

for i in range(n_det):
    psd_full = np.array(batched_det.PowerSpectralDensity[i])
    d_full   = np.array(batched_det.FrequencySeries[i])

    # Inner product weight: 4 * df / S_n
    inv_psd = 4.0 * df / psd_full

    # Linear weights: project PSD weights through basis interpolant
    w_lin_i = np.real(np.dot(np.conj(B_lin).T, inv_psd))
    roq_weights_lin.append(w_lin_i)

    # Quadratic weights
    w_qua_i = np.real(np.dot(np.conj(B_qua).T, inv_psd))
    roq_weights_qua.append(w_qua_i)

    # Project data: d_roq = B_lin^T @ (conj(d) * 4df/S_n)
    d_weighted = np.conj(d_full) * inv_psd
    d_lin_i = np.dot(np.conj(B_lin).T, d_weighted)
    data_lin_proj.append(d_lin_i)

roq_weights_lin = jnp.array(roq_weights_lin)
roq_weights_qua = jnp.array(roq_weights_qua)
data_lin_proj   = jnp.array(data_lin_proj)

# Precompute <d|d> (constant offset)
dd_terms = jnp.zeros(n_det)
for i in range(n_det):
    psd_i = batched_det.PowerSpectralDensity[i]
    d_i   = batched_det.FrequencySeries[i]
    dd = 4.0 * df * jnp.sum(jnp.abs(d_i)**2 / psd_i).real
    dd_terms = dd_terms.at[i].set(dd)

print(f"ROQ weights computed for {n_det} detectors.")
print(f"  Linear  weights shape : {roq_weights_lin.shape}")
print(f"  Quadratic weights shape: {roq_weights_qua.shape}")
print(f"  Data projection shape : {data_lin_proj.shape}")

In [ ]:
from sharpy.GW_likelihood import antenna_pattern_functions
from sharpy.utils import TimeDelayFromEarthCenter


def roq_single_detector_logL(params_13, det_idx,
                              roq_w_lin, roq_w_qua,
                              d_lin, dd,
                              batched_det, nodes_lin, nodes_qua):
    """ROQ log-likelihood for a single detector."""
    f_full_det = batched_det.Frequency[det_idx]

    # Generate template on full grid
    h_plus, h_cross = _gw_mod.template(params_13, f_full_det)

    # Antenna pattern + time delay
    lat  = batched_det.latitude[det_idx]
    lon  = batched_det.longitude[det_idx]
    gam  = batched_det.gamma[det_idx]
    zeta = batched_det.zeta[det_idx]
    elev = batched_det.elevation[det_idx]
    trig = batched_det.trigtime[det_idx]
    T    = batched_det.T[det_idx]

    fplus, fcross = antenna_pattern_functions(params_13, lat, lon, gam, zeta, trig)

    ra, dec = params_13[0], params_13[1]
    tc = trig + params_13[8]
    timedelay = TimeDelayFromEarthCenter(lat, lon, elev, ra, dec, tc)
    timeshift = timedelay + (params_13[8] + (T - 1))
    shift = 2.0 * jnp.pi * f_full_det * timeshift
    phase_shift = jnp.cos(shift) - 1j * jnp.sin(shift)

    # Full projected waveform
    h_full = (fplus * h_plus + fcross * h_cross) * phase_shift

    # Extract at ROQ nodes
    h_at_lin = h_full[nodes_lin]
    h_at_qua = jnp.abs(h_full[nodes_qua])**2

    # <d|h> via linear ROQ
    dh = jnp.sum(d_lin * h_at_lin).real

    # <h|h> via quadratic ROQ
    hh = jnp.sum(roq_w_qua * h_at_qua)

    # log L = <d|h> - 0.5 * <h|h> - 0.5 * <d|d>
    return dh - 0.5 * hh - 0.5 * dd


def log_likelihood_roq_full(params_13):
    """ROQ log-likelihood summed over all detectors."""
    logL = 0.0
    for i in range(n_det):
        logL += roq_single_detector_logL(
            params_13, i,
            roq_weights_lin[i], roq_weights_qua[i],
            data_lin_proj[i], dd_terms[i],
            batched_det, nodes_lin, nodes_qua,
        )
    return logL


def log_likelihood_roq_reduced(params_11):
    """Insert fixed RA/Dec and evaluate ROQ likelihood.

    params_11 layout:
        [0] logdist, [1] incl, [2] phic, [3] pol,
        [4] mc, [5] q, [6] tc, [7] chi1, [8] chi2,
        [9] lambda_1, [10] lambda_2
    """
    params_13 = jnp.concatenate([
        jnp.array([FIXED_RA, FIXED_DEC]),
        params_11[:4],
        params_11[4:9],
        params_11[9:11],
    ])
    return log_likelihood_roq_full(params_13)

print("ROQ likelihood functions defined.")

In [ ]:
prior_bounds = jnp.array([
    [jnp.log(1.0),  jnp.log(75.0)],     # [0]  logdistance
    [0.0,           jnp.pi],             # [1]  inclination
    [0.0,           2 * jnp.pi],         # [2]  phic
    [0.0,           jnp.pi],             # [3]  pol
    [1.18,          1.21],               # [4]  mc
    [0.5,           1.0],                # [5]  q
    [-0.1,          0.1],                # [6]  tc
    [-0.5,          0.5],                # [7]  chi1
    [-0.5,          0.5],                # [8]  chi2
    [5.0,           5000.0],             # [9]  lambda_1
    [5.0,           5000.0],             # [10] lambda_2
])

boundary_conditions = jnp.array([0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0])

parameter_names = [
    "logdistance", "theta_jn", "phiref", "pol",
    "mc", "q", "tc", "chi1", "chi2", "lambda_1", "lambda_2",
]


def prior(params):
    return 0.0

print(f"Sampling {len(parameter_names)} parameters: {parameter_names}")

## Sanity check

Compare the ROQ likelihood against the standard (full-grid) likelihood
at a few test points to verify correctness.

In [ ]:
log_L_roq = jax.jit(log_likelihood_roq_reduced)
log_L_std = jax.jit(partial(log_likelihood_det, detector_list=batched_det))

p_test = jnp.array([
    jnp.log(40.0), 2.5, 0.0, 0.0,
    1.186, 0.87, 0.0, 0.0, 0.0, 300.0, 300.0
])

# Warm-up
_ = log_L_roq(p_test).block_until_ready()
p13_test = jnp.concatenate([jnp.array([FIXED_RA, FIXED_DEC]), p_test[:4], p_test[4:9], p_test[9:11]])
_ = log_L_std(p13_test).block_until_ready()

# Compare
header = f"{'Parameter':>50s}  {'ROQ logL':>12s}  {'Std logL':>12s}  {'Diff':>10s}"
print(header)
print("-" * 90)

import gc
for mc in [1.186, 1.195, 1.200]:
    for q_val in [0.87, 1.0]:
        for logd in [jnp.log(30.), jnp.log(40.)]:
            p11 = jnp.array([logd, 2.5, 0.0, 0.0, mc, q_val, 0.0, 0.0, 0.0, 300.0, 300.0])
            p13 = jnp.concatenate([jnp.array([FIXED_RA, FIXED_DEC]), p11[:4], p11[4:9], p11[9:11]])
            ll_roq = float(log_L_roq(p11))
            ll_std = float(log_L_std(p13))
            diff = ll_roq - ll_std
            desc = f"mc={mc:.3f}, q={q_val:.2f}, D={float(jnp.exp(logd)):.0f}"
            print(f"{desc:>50s}  {ll_roq:>12.2f}  {ll_std:>12.2f}  {diff:>10.4f}")
gc.collect()

# Timing
import timeit
n_eval = 100
t_roq = timeit.timeit(lambda: log_L_roq(p_test).block_until_ready(), number=n_eval)
t_std = timeit.timeit(lambda: log_L_std(p13_test).block_until_ready(), number=n_eval)
print(f"\nTiming ({n_eval} evaluations):")
print(f"  Standard : {1000*t_std/n_eval:.2f} ms/eval")
print(f"  ROQ      : {1000*t_roq/n_eval:.2f} ms/eval")
print(f"  Speedup  : {t_std/t_roq:.1f}x")

In [ ]:
N_PARTICLES = 500
STEP_SIZE   = 0.3
ALPHA       = 0.95
SEED        = 42

print(f"Starting SHARPy SMC with ROQ likelihood...")
print(f"  {N_PARTICLES} particles, {len(parameter_names)} parameters")
start = time.time()

result_dict = run_sharpy(
    log_likelihood_roq_reduced, prior,
    prior_bounds, boundary_conditions,
    ALPHA, N_PARTICLES, STEP_SIZE,
    jax.random.PRNGKey(SEED),
    folder=OUTDIR, label=LABEL,
)

dt = time.time() - start
samples = result_dict["posterior_samples"]
logZ, dlogZ = result_dict["logZ"], result_dict["dlogZ"]
print(f"\nDone in {dt:.1f}s ({dt/3600:.2f}h)")
print(f"log Z = {logZ:.2f} +/- {dlogZ:.2f}")
print(f"Posterior samples: {samples.shape}")

In [ ]:
from corner import corner

fig = corner(
    np.array(samples), show_titles=True,
    labels=parameter_names, title_kwargs={"fontsize": 10},
)
plot_path = os.path.join(OUTDIR, f"{LABEL}_corner.png")
fig.savefig(plot_path, dpi=150)
print(f"Saved: {plot_path}")
fig

In [ ]:
from sharpy.utils import McQ2Masses

mc_samples   = np.array(samples[:, 4])
q_samples    = np.array(samples[:, 5])
chi1_samples = np.array(samples[:, 7])
chi2_samples = np.array(samples[:, 8])
lam1_samples = np.array(samples[:, 9])
lam2_samples = np.array(samples[:, 10])
logd_samples = np.array(samples[:, 0])

m1_samples = np.zeros(len(mc_samples))
m2_samples = np.zeros(len(mc_samples))
for i in range(len(mc_samples)):
    m1_samples[i], m2_samples[i] = McQ2Masses(mc_samples[i], q_samples[i])

chi_eff_samples = (m1_samples * chi1_samples + m2_samples * chi2_samples) / (m1_samples + m2_samples)

M_samples = m1_samples + m2_samples
lambda_tilde_samples = (16.0 / 13.0) * (
    (m1_samples + 12.0 * m2_samples) * m1_samples**4 * lam1_samples
    + (m2_samples + 12.0 * m1_samples) * m2_samples**4 * lam2_samples
) / M_samples**5

dL_samples = np.exp(logd_samples)

paper_samples = np.column_stack([
    mc_samples, q_samples, chi_eff_samples,
    lambda_tilde_samples, dL_samples,
])

paper_labels = [
    r"$\mathcal{M}_c$ $[M_\odot]$",
    r"$q$",
    r"$\chi_{\rm eff}$",
    r"$\tilde{\Lambda}$",
    r"$D_L$ [Mpc]",
]

fig = corner(
    paper_samples, show_titles=True,
    labels=paper_labels,
    title_kwargs={"fontsize": 12},
    quantiles=[0.05, 0.5, 0.95],
    levels=(0.5, 0.9),
    fill_contours=True,
    color="tab:orange",
)
plot_path = os.path.join(OUTDIR, f"{LABEL}_corner_paper.png")
fig.savefig(plot_path, dpi=150)
print(f"Saved: {plot_path}")
fig